# 03. Fenêtrage et extraction de features

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # racine du projet
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, io, viz, models, features
from src import postprocess as pp, evaluation as ev
pd.set_option("display.width", 200)
print("Dossier données :", config.DATA_DIR)

Objectif : découper le signal en fenêtres (2 s, recouvrement 50 %) et extraire des features par canal.

> ℹ️ **Note d'architecture — Fenêtrage & Pipeline de Features :**
> L'extraction (src.features.build_feature_dataset()) orchestre automatiquement le fenêtrage via src.windowing.window_starts(), l'étiquetage via src.labeling, et le pré-traitement du signal via src.preprocess. Les caractéristiques (temporelles, spectrales, Hjorth, entropie de permutation, ondelettes et synchronie inter-canaux) sont vectorisées par lots et mises en cache Parquet (cache/features.parquet).

In [ ]:
feats = features.build_feature_dataset()  # utilise le cache si présent
print(feats.shape)
print("Taux de fenêtres crises :", round(100*feats.label.mean(), 2), "%")
feats[["patient", "t_start", "label"]].head()

Features extraites par canal : statistiques (moyenne, écart-type, percentiles), énergie de bandes (delta/theta/alpha/beta/gamma), ratios d'asymétrie, entropie spectrale, entropie de permutation et énergie en ondelettes. Les canaux manquants sont marqués par un masque.

In [ ]:
import re
familles = pd.Series([re.sub(r'^EEG[A-Za-z0-9]+_', '', c) for c in feats.columns if c.startswith('EEG')])
familles.value_counts().head(20)